# 01 — Data Cleaning

This notebook loads the raw AHI data from Excel, fixes header rows, and exports each sheet as a clean CSV for downstream analysis.

**Data source:** [Shimberg Center Assisted Housing Inventory](http://flhousingdata.shimberg.ufl.edu/assisted-housing-inventory/results?nid=100) — Alachua County, FL.

**Missing data conventions:** When a value is missing in a numeric column, we use `NaN` so the column stays numeric and missing values are automatically excluded from calculations. For text or categorical columns, we use `N/A`. Columns that use `x` and `-` to indicate whether something applies are converted to `True` and `False`.

In [1]:
import pandas as pd

file_path = "../Data/Raw Data/AHI_Raw_Data.xlsx"

df_funder = pd.read_excel(file_path, sheet_name="Sheet 1")
df_inventory = pd.read_excel(file_path, sheet_name="Sheet 2")
df_assistance_program = pd.read_excel(file_path, sheet_name="Sheet 3")
df_lost_property = pd.read_excel(file_path, sheet_name="Sheet 4")
df_housing_agencies = pd.read_excel(file_path, sheet_name="Sheet 5")

print("df_funder:", df_funder.shape)
print("df_inventory:", df_inventory.shape)
print("df_assistance_program:", df_assistance_program.shape)
print("df_lost_property:", df_lost_property.shape)
print("df_housing_agencies:", df_housing_agencies.shape)

df_funder: (8, 7)
df_inventory: (60, 168)
df_assistance_program: (10, 38)
df_lost_property: (11, 27)
df_housing_agencies: (4, 13)


### Header Fix

All 5 sheets contain a title row followed by a blank row before the actual column headers. We re-read each sheet using `header=2` to skip those two rows and set the third row as the proper header.

In [2]:
df_funder = pd.read_excel(file_path, sheet_name="Sheet 1", header=2)
df_inventory = pd.read_excel(file_path, sheet_name="Sheet 2", header=2)
df_assistance_program = pd.read_excel(file_path, sheet_name="Sheet 3", header=2)
df_lost_property = pd.read_excel(file_path, sheet_name="Sheet 4", header=2)
df_housing_agencies = pd.read_excel(file_path, sheet_name="Sheet 5", header=2)

print("df_funder:", df_funder.shape)
print("df_inventory:", df_inventory.shape)
print("df_assistance_program:", df_assistance_program.shape)
print("df_lost_property:", df_lost_property.shape)
print("df_housing_agencies:", df_housing_agencies.shape)

df_funder: (6, 7)
df_inventory: (58, 168)
df_assistance_program: (8, 38)
df_lost_property: (9, 27)
df_housing_agencies: (2, 13)


### df_funder

No cleaning needed — all values are complete and well-formed.

### df_inventory

Columns with `-`, `not avail.`, or `0` as placeholders are replaced based on type:
- **`N/A`** (categorical/text): ID columns (`FHFC Key`, `HUD REMS`, `Public Housing Development #`), location and program columns (`Florida DOR Parcel` through `Housing Programs`), and `Target Population`, `Occupancy Status`, `Owner Type`, `Developer Name and Contact Information`, `Owner Name and Contact information`, `Construction Type (FHFC Only)'`. Note: `"Construction Category Not Found"` is kept as-is — it is a real source value, not a placeholder.
- **`True`/`False`** (binary flags): `FHFC Funded` through `LHFA Funded`, `Has Housing Credits 4%` through `Has Local Bonds`, and `FHFC Preservation Set-Aside`. `In QCT 2022` and `In DDA 2022` use `Y`/`N` and are converted to `Yes`/`No`.
- **`NaN`** (numeric, not applicable): funding year and expiration date columns (`HC 4% Funding Year` through `Local Bonds Maturity Date`), `REAC Score (HUD only)` and `REAC Inspection Date` (non-HUD properties), `HUD/RD Rental Assistance Units`, `Public Housing ACC Units`, `Total Living Area`, `Number of Buildings`, `Last Property Sale Price ($)`, `Last Property Sale Date`, `Just Value ($)`, `Occupancy Rate`, and average rent columns (`0 BR Av. Rent ($)` through `4+ BR Av. Rent ($)`) — a value of 0 in a rent column means the development has no units of that bedroom size, not that rent is free.

Tenant demographic columns (`Average Household Size (Persons)` through `# of Households Reporting`) use `-1` for missing values, replaced with `NaN`. Codes `-4` (suppressed, <11 households) and `-5` (non-reporting, <50% rate) are documented AHI codes but do not appear in this dataset.

Note: REAC scores may include a letter suffix (e.g., `83b`) indicating a health and safety deficiency level.

In [3]:
import numpy as np

# ID columns: missing values indicate property not in that system
id_cols = df_inventory.loc[:, "FHFC Key":"Public Housing Development #"].columns
df_inventory[id_cols] = df_inventory[id_cols].replace({"-": "N/A"})

# Text columns: various placeholders → N/A
text_cols = df_inventory.loc[:, "Florida DOR Parcel":"Housing Programs"].columns
df_inventory[text_cols] = df_inventory[text_cols].replace({"-": "N/A", "not avail.": "N/A", 0: "N/A"})

cat_cols = ["Target Population", "Occupancy Status", "Owner Type",
            "Developer Name and Contact Information", "Owner Name and Contact information",
            "Construction Type (FHFC Only)'"]
df_inventory[cat_cols] = df_inventory[cat_cols].replace({"-": "N/A", "not avail.": "N/A"})

# Blanket replace: remaining x/- flags → 1/0
df_inventory = df_inventory.replace({"-": 0, "x": 1, "not avail.": 0})

# Funding flag and program presence columns: 0/1 → True/False
bool_cols = (df_inventory.loc[:, "FHFC Funded":"LHFA Funded"].columns.tolist() +
             df_inventory.loc[:, "Has Housing Credits 4%":"Has Local Bonds"].columns.tolist() +
             ["FHFC Preservation Set-Aside"])
df_inventory[bool_cols] = df_inventory[bool_cols].astype(bool)

# QCT/DDA designation: Y/N → Yes/No
df_inventory[["In QCT 2022", "In DDA 2022"]] = df_inventory[["In QCT 2022", "In DDA 2022"]].replace({"Y": "Yes", "N": "No"})

# Funding year and expiration date columns: 0 → NaN (year 0 is not meaningful)
year_cols = df_inventory.loc[:, "HC 4% Funding Year":"Local Bonds Maturity Date"].columns
df_inventory[year_cols] = df_inventory[year_cols].replace(0, np.nan)

# REAC columns: 0 → NaN for non-HUD properties
df_inventory["REAC Score (HUD only)"] = df_inventory["REAC Score (HUD only)"].replace(0, np.nan)
df_inventory["REAC Inspection Date"] = df_inventory["REAC Inspection Date"].replace(0, np.nan)

# Numeric not-applicable columns: 0 → NaN
na_numeric_cols = ["HUD/RD Rental Assistance Units", "Public Housing ACC Units",
                   "Total Living Area", "Number of Buildings",
                   "Last Property Sale Price ($)", "Last Property Sale Date", "Just Value ($)",
                   "Occupancy Rate",
                   "0 BR Av. Rent ($)", "1 BR Av. Rent ($)", "2 BR Av. Rent ($)",
                   "3 BR Av. Rent ($)", "4+ BR Av. Rent ($)"]
df_inventory[na_numeric_cols] = df_inventory[na_numeric_cols].replace(0, np.nan)

# Tenant demographic columns: -1 (missing) → NaN
tenant_cols = df_inventory.loc[:, "Average Household Size (Persons)":"# of Households Reporting"].columns
df_inventory[tenant_cols] = df_inventory[tenant_cols].replace(-1, np.nan)

df_inventory.head()

C:\Users\yimol\AppData\Local\Temp\ipykernel_24532\1198663466.py:17: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_inventory = df_inventory.replace({"-": 0, "x": 1, "not avail.": 0})


,Shim ID,FHFC Key,HUD REMS,Public Housing Development #,Florida DOR Parcel,Development Name,Street Address,City,Zip Code,County,...,"% of Households with at least one child age 0-17, 2018-2022, Tract","% of Households with at least one child age 0-17, 2018-2022, County","% of Households with at least one person age 65 or older, 2018-2022, Tract","% of Households with at least one person age 65 or older, 2018-2022, County",Total Living Area,Number of Buildings,REAC Score (HUD only),REAC Inspection Date,Construction Type (FHFC Only)',In unincorporated area
0,1213,N/A,800003897,N/A,3926002000,Alachua Apts,13605 NW County Road 235,Alachua,32615,Alachua,...,19.46,22.34,32.83,26.18,56979.0,15.0,96,2025-05-20,N/A,NaN
1,112,N/A,N/A,N/A,3563000000,Alachua Villas,14000 NW 154TH AVE,Alachua,32615,Alachua,...,34.30,22.34,38.12,26.18,26660.0,9.0,NaN,NaN,N/A,NaN
2,7872,3308,N/A,N/A,3214035000,Arbours at Merrillwood I,13207 NW 153rd Place,Alachua,32615,Alachua,...,39.30,22.34,25.72,26.18,NaN,NaN,NaN,NaN,N/A,NaN
3,7999,3454,N/A,N/A,3926002000,Sherwood Oaks,13605 NW County Road 235,Alachua,32615,Alachua,...,19.46,22.34,32.83,26.18,NaN,NaN,NaN,NaN,Construction Category Not Found,NaN
4,1356,N/A,800004435,N/A,3926020000,Sherwood Oaks Apartments,13400 NW 140th St,Alachua,32615,Alachua,...,19.46,22.34,32.83,26.18,48982.0,11.0,83b,2018-11-19,N/A,NaN


### df_assistance_program

Some rows have `0` in the Address and State columns instead of valid values. Since all entries in this dataset are located in Gainesville, Alachua County, we replace the `0` values in State with `FL` and mark missing addresses as `N/A`.

All columns that use `x` and `-` to indicate a yes/no relationship are converted to `True` and `False`.

In [4]:
df_assistance_program["Address"] = df_assistance_program["Address"].replace(0, "N/A")
df_assistance_program["State"] = df_assistance_program["State"].replace(0, "FL")

funding_cols = df_assistance_program.columns[df_assistance_program.columns.get_loc("McKinney Vento-funded"):]
df_assistance_program[funding_cols] = df_assistance_program[funding_cols].replace({"-": False, "x": True})

df_assistance_program.head()

C:\Users\yimol\AppData\Local\Temp\ipykernel_24532\5667861.py:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_assistance_program[funding_cols] = df_assistance_program[funding_cols].replace({"-": False, "x": True})


,Project Name,Address,City,State,Zip,County,Continuum of Care,Provider Name,Project Type,Site or Tenant Based,...,VA Grant and Per Diem Program,VA Community Contract Safe Haven Program,VA Contract Residential Services,VA Community Contract Safe Haven,HHS Runaway and Homeless Youth Basic Center Program,HHS Runaway and Homeless Youth Transitional Living Program,HHS Runaway and Homeless Youth Maternity Group Homes,HHS Runaway and Homeless Demonstration Project,HOPWA,Other Federal Funding.1
0,ACCHH Permanent Supportive Housing (New HUD ...,3055 NE 28th Drive,Gainesville,FL,32609,Alachua,"Gainesville/Alachua, Putnam Counties CoC",ACCHH Alachua County Coalition for the Hungr...,Permanent Supportive Housing,Tenant-based - scattered site,...,False,False,False,False,False,False,False,False,False,False
1,ACCHH Permanent Supportive Housing Program (...,3055 NE 28th Drive,Gainesville,FL,32609,Alachua,"Gainesville/Alachua, Putnam Counties CoC",ACCHH Alachua County Coalition for the Hungr...,Permanent Supportive Housing,Tenant-based - scattered site,...,False,False,False,False,False,False,False,False,False,False
2,ACHA HUD VASH,703 NE 1st Street,Gainesville,FL,32609,Alachua,"Gainesville/Alachua, Putnam Counties CoC",ACHA HUD VASH,Permanent Supportive Housing,Tenant-based - scattered site,...,False,False,False,False,False,False,False,False,False,False
3,Family Promise Permanent Supportive Housing ...,N/A,Gainesville,FL,32601,Alachua,"Gainesville/Alachua, Putnam Counties CoC",Family Promise Gainesville AGENCY,Permanent Supportive Housing,Tenant-based - scattered site,...,False,False,False,False,False,False,False,False,False,False
4,Family Promise Rapid Rehousing (ESG) - FL-508,N/A,Gainesville,FL,32601,Alachua,"Gainesville/Alachua, Putnam Counties CoC",Family Promise Emergency Shelter - FL-508,Rapid Rehousing,Tenant-based - scattered site,...,False,False,False,False,False,False,False,False,False,False


### df_lost_property

The `FHFC HPP key` and `HUD/RD Rental Assistance Units` columns use `-` for properties where the value doesn't apply. We replace these with `NaN` instead of `0` or a string like `"N/A"` so the columns stay numeric, missing values are automatically excluded from aggregations, and there's no confusion with a real value of zero.

The `Approximate Year of Loss` column lists `9999` as the year for Gardens of Gainesville, which we assume is a typo for `1999`.

The `Year Built/Funded`, `Type of Ownership`, and `Population Served` columns contain `not avail.` for properties where the information is unknown. We replace these with `N/A` for consistency.

All columns that use `x` and `-` to indicate a yes/no relationship are converted to `True` and `False`.

In [5]:
import numpy as np

pd.set_option('display.max_columns', None)

df_lost_property["FHFC HPP key"] = df_lost_property["FHFC HPP key"].replace("-", np.nan)
df_lost_property["HUD/RD Rental Assistance Units"] = df_lost_property["HUD/RD Rental Assistance Units"].replace({"-": np.nan, " - ": np.nan})
df_lost_property["Approximate Year of Loss"] = df_lost_property["Approximate Year of Loss"].replace(9999, 1999)
df_lost_property["Year Built/Funded"] = df_lost_property["Year Built/Funded"].replace("not avail.", "N/A")
df_lost_property["Type of Ownership"] = df_lost_property["Type of Ownership"].replace("not avail.", "N/A")
df_lost_property["Population Served"] = df_lost_property["Population Served"].replace("not avail.", "N/A")

bool_cols = df_lost_property.columns[df_lost_property.columns.get_loc("FHFC Funded"):df_lost_property.columns.get_loc("RD Prepaid Mortgage") + 1]
df_lost_property[bool_cols] = df_lost_property[bool_cols].replace({"-": False, "x": True})

df_lost_property.head()

C:\Users\yimol\AppData\Local\Temp\ipykernel_24532\287211764.py:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_lost_property["FHFC HPP key"] = df_lost_property["FHFC HPP key"].replace("-", np.nan)
C:\Users\yimol\AppData\Local\Temp\ipykernel_24532\287211764.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_lost_property["HUD/RD Rental Assistance Units"] = df_lost_property["HUD/RD Rental Assistance Units"].replace({"-": np.nan, " - ": np.nan})
C:\Users\yimol\AppData\Local\Temp\ipykernel_24532\287211764.py:13: FutureWarning:

,LPI ID,Shim ID,FHFC HPP key,Name,Address,City,Zip,County,Total Units,Estimated Assisted Units,HUD/RD Rental Assistance Units,Housing Program(s),FHFC Funded,RD Funded,LHFA Funded,HUD Funded,Approximate Year of Loss,Tax Credit or Bond Property No Longer Under Use Restrictions,HUD Section 8 Expired Contract/Opt-Out,HUD Prepaid Mortgage,HUD Assigned Mortgage,RD Prepaid Mortgage,Year Built/Funded,Type of Ownership,Population Served,Latitude,Longitude
0,15,1211,144.0,Cedar Ridge,412 Southwest 69th Street,Gainesville,32607,Alachua,8,3,NaN,Federal Deposit Insurance Corporation,True,False,False,False,1999,False,False,False,False,False,1995,N/A,N/A,29.648517,-82.416275
1,310,1000310,287.0,Gardens of Gainesville,75 SW 75 Street,Gainesville,32607,Alachua,27,5,NaN,Rental Assistance/HUD,True,False,False,False,1999,False,True,False,False,False,1995,N/A,N/A,29.652225,-82.421441
2,433,1310,NaN,Glen Springs Manor,2130 NW 31st Avenue,Gainesville,32605,Alachua,136,136,136.0,Rental Assistance/HUD,False,False,False,True,2009,False,True,True,False,False,1970,For-Profit,Family,29.683970,-82.352424
3,822,1060,259.0,Savannah,316 S.W. 62nd Boulevard,Gainesville,32607,Alachua,178,178,NaN,State Bonds,True,False,False,False,2021,False,False,False,False,False,2001,For-Profit,Family,29.649383,-82.408522
4,443,1357,NaN,Seminary Lane,1019 NW 5th Ave,Gainesville,32601,Alachua,53,53,53.0,Rental Assistance/HUD,False,False,False,True,2009,False,True,False,False,False,1979,Other,Family,29.656199,-82.337789


### df_housing_agencies

Gainesville Housing Authority has `0` for `Capital Fund`, which is likely a data entry issue.

> **Note:** We leave this value as-is since we don't have a reliable replacement, but it should be treated with caution in any analysis involving capital funding.

In [6]:
df_housing_agencies.head()

,Name,Address,City,Zip Code,County,Phone Number,Public Housing Units,Housing Choice Vouchers,PHA Size,Performance Designation,Occupancy Rate,Operating Subsidy,Capital Fund
0,Alachua County,703 NE 1st St ...,Gainesville,32601,Alachua,3523722549,257,1160,257,Standard Performer,75.6,868493,1071997
1,Gainesville Housing Authority,1900 SE 4th St ...,Gainesville,32641,Alachua,3528725502,574,1914,640,Standard Performer,67.0,1519139,0


### Export Cleaned Data

We export all 5 cleaned dataframes to CSV so they can be loaded directly in the EDA notebooks without repeating the cleaning steps.

In [7]:
df_funder.to_csv("../Data/Cleaned Data/df_funder.csv", index=False)
df_inventory.to_csv("../Data/Cleaned Data/df_inventory.csv", index=False)
df_assistance_program.to_csv("../Data/Cleaned Data/df_assistance_program.csv", index=False)
df_lost_property.to_csv("../Data/Cleaned Data/df_lost_property.csv", index=False)
df_housing_agencies.to_csv("../Data/Cleaned Data/df_housing_agencies.csv", index=False)